# 02 - Tiny PPO Training Watch

**Goal:** Run a very small PPO job and inspect the signals it produces.

**What you will learn:** How smoke training writes checkpoints/progress files and how to compare behavior before and after training.

**Inputs:** An installed RL stack with Ray/RLlib/SoccerTwos; no prior checkpoint is required.

**Outputs:** A tiny checkpoint when training succeeds, progress tables, training curves, and before/after rollout plots.

**Success criteria:** The notebook either trains a tiny policy or clearly explains what dependency/artifact is missing.

In [ ]:
from pathlib import Path
import importlib
import os
import sys

PROJECT_MARKER = Path("soccer_twos_project") / "notebook_tools.py"


def _running_in_colab():
    if "google.colab" in sys.modules:
        return True
    if os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_GPU"):
        return True
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _candidate_project_roots():
    seen = set()

    def add(path):
        path = Path(path).expanduser()
        key = str(path)
        if key not in seen:
            seen.add(key)
            yield path

    for env_name in ("SOCCER_TWOS_PROJECT_ROOT", "PROJECT_ROOT"):
        value = os.environ.get(env_name)
        if value:
            yield from add(value)

    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        yield from add(base)
        yield from add(base / "soccer-twos-starter")
        yield from add(base / "project" / "soccer-twos-starter")

    if sys.platform == "darwin":
        yield from add(
            Path.home()
            / "all_data"
            / "Georgia Tech"
            / "Course Content"
            / "CS 8803- DRL"
            / "project"
            / "soccer-twos-starter"
        )

    if _running_in_colab():
        try:
            from google.colab import drive  # type: ignore
            if not Path("/content/drive/MyDrive").exists():
                drive.mount("/content/drive")
        except Exception:
            pass
        for drive_root in (Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives"), Path("/content")):
            for relative in (
                Path("CS 8803- DRL") / "project" / "soccer-twos-starter",
                Path("project") / "soccer-twos-starter",
                Path("soccer-twos-starter"),
                Path("Colab Notebooks") / "soccer-twos-starter",
            ):
                yield from add(drive_root / relative)


def _find_project_root():
    for candidate in _candidate_project_roots():
        if (candidate / PROJECT_MARKER).exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find soccer_twos_project/notebook_tools.py. "
        "Open this notebook from the project root/notebooks folder, or set SOCCER_TWOS_PROJECT_ROOT."
    )


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for _module_name in list(sys.modules):
    if _module_name == "soccer_twos_project" or _module_name.startswith("soccer_twos_project."):
        del sys.modules[_module_name]

importlib.invalidate_caches()
from IPython.display import Markdown, display
from soccer_twos_project.notebook_tools import *

ctx = setup_project()
show_hardware()

LEARNING_DIR = learning_artifact_dir(ctx, "visual_learning")
print("Learning artifacts:", LEARNING_DIR)

## Configuration

This is not performance training. Keep the timestep count small so the mechanics are fast to inspect.

In [ ]:
RUN_TINY_TRAINING = True
TINY_TIMESTEPS = 3_000
PRE_STEPS = 150
POST_STEPS = 150
RUN_TENSORBOARD = False

## Before Training: Random Behavior

This gives a baseline for action mix, distances, and reward before PPO updates anything.

In [ ]:
random_before = None
try:
    random_before = collect_single_player_rollout(
        policy="random",
        steps=PRE_STEPS,
        render=False,
        label="before training: random",
    )
    display(rollout_summary_table({"before random": random_before}))
    plot_rollout_overview(random_before, title="Before training: random policy");
except Exception as exc:
    print("Random baseline rollout skipped:", type(exc).__name__, exc)

## Run Tiny PPO

This cell should create a Ray trial folder, `progress.csv`, `run_metadata.json`, and a checkpoint. If the local kernel is not the `soccertwos` environment, it will fail with an actionable dependency message.

In [ ]:
tiny_checkpoint = None

if RUN_TINY_TRAINING:
    try:
        tiny_checkpoint = run_training(
            ctx,
            stage="ppo_baseline",
            profile_name="cpu_debug",
            timesteps=TINY_TIMESTEPS,
            smoke=True,
            checkpoint_freq=1,
            verbose=1,
        )
    except Exception as exc:
        print("Tiny training failed:", type(exc).__name__, exc)
        print("Activate the soccertwos environment, then rerun this notebook.")
else:
    print("RUN_TINY_TRAINING=False. Reusing any existing ppo_baseline checkpoint if available.")

if tiny_checkpoint is None:
    try:
        tiny_checkpoint = best_checkpoint(ctx, "ppo_baseline")
        print("Using existing checkpoint:", tiny_checkpoint)
    except Exception as exc:
        print("No existing checkpoint found:", type(exc).__name__, exc)

print("Tiny checkpoint:", tiny_checkpoint)

## TensorBoard Guidance

TensorBoard reads the same Ray folders as `progress.csv`. Use the launcher when you want a browser view.

In [ ]:
print("TensorBoard logdir:", ctx.dirs["checkpoints"])
if RUN_TENSORBOARD:
    tensorboard_proc = launch_tensorboard(ctx, port=6006)
else:
    print("Set RUN_TENSORBOARD=True to launch TensorBoard at http://localhost:6006.")

## Training Diagnostics

Reward can stay flat in a tiny run. Check that progress rows exist, episode length is finite, and PPO diagnostics are being written.

In [ ]:
progress_df = load_progress_table(ctx, "ppo_baseline")
if progress_df.empty:
    print("No progress table yet. Run the tiny training cell first.")
else:
    display(progress_status(ctx, "ppo_baseline", rows=10))
    plot_training_diagnostics(progress_df, title="Tiny PPO training diagnostics");
    print_json(checkpoint_summary(ctx, "ppo_baseline"))

## After Training: Roll Out The Tiny Checkpoint

Tiny training may still be weak. The useful question is whether behavior changed at all.

In [ ]:
trained_after = None
if tiny_checkpoint:
    try:
        trained_after = collect_checkpoint_rollout(
            tiny_checkpoint,
            stage="ppo_baseline",
            steps=POST_STEPS,
            render=False,
            label="after tiny PPO",
        )
        rollouts = {"before random": random_before, "after tiny PPO": trained_after}
        display(rollout_summary_table({key: value for key, value in rollouts.items() if value is not None}))
        plot_rollout_comparison(rollouts, title="Behavior metrics before and after tiny PPO");
        plot_action_distribution(random_before, title="Before training action distribution");
        plot_action_distribution(trained_after, title="After tiny PPO action distribution");
    except Exception as exc:
        print("Checkpoint rollout skipped:", type(exc).__name__, exc)
else:
    print("No checkpoint available yet. Run the tiny training cell first.")

## Key Takeaways

A smoke run is successful if it creates progress rows and a checkpoint. Performance may still be poor; this notebook is for proving that the training loop works and that behavior can be inspected.

## What To Run Next

Run `03_behavior_before_after_training.ipynb` after you have a checkpoint or exported package to compare trained behavior more carefully.